In [4]:
#1.Importing Libraries
import pandas as pd
import numpy as np
import zipfile
import joblib

from google.colab import files

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# Set random seed for reproducibility
np.random.seed(42)

In [5]:
#2.Uploading and Loading the Dataset
uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall()

df = pd.read_csv("riyadh_resturants_clean.csv")
print("Original dataset shape:", df.shape)
df.head()



Saving riyadh_resturants_clean.csv.zip to riyadh_resturants_clean.csv (2).zip
Original dataset shape: (19361, 11)


,name,categories,address,lat,lng,price,likes,photos,tips,rating,ratingSignals
0,مطعم وقت الشواية,Afghan Restaurant,"الرياض 14723, المملكة العربية السعودية",24.518983,46.656981,Moderate,0.0,0,0,NaN,NaN
1,ديوانية عكاظ,Café,"الرياض 14726, المملكة العربية السعودية",24.518511,46.669149,Cheap,0.0,2,0,NaN,NaN
2,شاهي جمر راعي الجمس,Coffee Shop,"الرياض 14726, المملكة العربية السعودية",24.519314,46.670041,Cheap,0.0,0,0,NaN,NaN
3,غدير الشام,Afghan Restaurant,المملكة العربية السعودية,24.519520,46.671660,Moderate,0.0,0,0,NaN,NaN
4,Dunkin',Donut Shop,"الرياض, المملكة العربية السعودية",24.525001,46.433944,Cheap,29.0,90,1,8.9,32.0


In [6]:
 #3.Cleaning the Dataset
restaurants = df.copy()

restaurants["categories"] = restaurants["categories"].fillna("")

restaurants["price"] = restaurants["price"].fillna("Unknown")

restaurants["rating"] = pd.to_numeric(
    restaurants["rating"],
    errors="coerce"
)

restaurants["likes"] = restaurants["likes"].fillna(0)

# Removed ratingSignals as it's not needed
restaurants = restaurants.drop(
    columns=["ratingSignals"],
    errors="ignore"
)

# Remove duplicate restaurants
restaurants = restaurants.drop_duplicates()


restaurants["categories"] = (
    restaurants["categories"]
    .str.lower()
    .str.replace(",", " ", regex=False)
    .str.strip()
)

print("Cleaned dataset shape:", restaurants.shape)

restaurants.head()

Cleaned dataset shape: (19360, 10)


,name,categories,address,lat,lng,price,likes,photos,tips,rating
0,مطعم وقت الشواية,afghan restaurant,"الرياض 14723, المملكة العربية السعودية",24.518983,46.656981,Moderate,0.0,0,0,NaN
1,ديوانية عكاظ,café,"الرياض 14726, المملكة العربية السعودية",24.518511,46.669149,Cheap,0.0,2,0,NaN
2,شاهي جمر راعي الجمس,coffee shop,"الرياض 14726, المملكة العربية السعودية",24.519314,46.670041,Cheap,0.0,0,0,NaN
3,غدير الشام,afghan restaurant,المملكة العربية السعودية,24.519520,46.671660,Moderate,0.0,0,0,NaN
4,Dunkin',donut shop,"الرياض, المملكة العربية السعودية",24.525001,46.433944,Cheap,29.0,90,1,8.9


In [7]:
#5.Understanding the Restaurant Data

print("Price distribution:")
print(restaurants["price"].value_counts())

print("\nRating statistics:")
print(restaurants["rating"].describe())

print("\nNumber of restaurants with ratings:")
print(restaurants["rating"].notna().sum())

print("\nNumber of restaurants without ratings:")
print(restaurants["rating"].isna().sum())

Price distribution:
price
Cheap             13499
Moderate           3980
Unknown            1515
Expensive           303
Very Expensive       63
Name: count, dtype: int64

Rating statistics:
count    7948.000000
mean        7.536047
std         0.942885
min         4.400000
25%         6.900000
50%         7.600000
75%         8.200000
max         9.600000
Name: rating, dtype: float64

Number of restaurants with ratings:
7948

Number of restaurants without ratings:
11412


In [8]:
#6.TF-IDF Restaurant Profiles

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(
    restaurants["categories"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (19360, 294)


In [9]:
#7 Define Example User Preferences

user_preferences = {
    "cuisine": "coffee shop",
    "budget": "Cheap",
    "minimum_rating": 7
}

print("User preferences:")
print(user_preferences)

User preferences:
{'cuisine': 'coffee shop', 'budget': 'Cheap', 'minimum_rating': 7}


In [10]:
#8.Calculating Content-Based Features

user_cuisine = tfidf.transform(
    [user_preferences["cuisine"]]
)

#Comparing the user's cuisine preference with every restaurant

cuisine_similarity = cosine_similarity(
    user_cuisine,
    tfidf_matrix
).flatten()

restaurants["cuisine_similarity"] = cuisine_similarity

#To check whether restaurant price matches user budget
restaurants["budget_match"] = (
    restaurants["price"] == user_preferences["budget"]
).astype(int)

#To check whether restaurant rating meets the user's minimum rating.


restaurants["rating_match"] = (
    restaurants["rating"].notna() &
    (restaurants["rating"] >= user_preferences["minimum_rating"])
).astype(int)

print(
    restaurants[
        [
            "name",
            "categories",
            "price",
            "rating",
            "cuisine_similarity",
            "budget_match",
            "rating_match"
        ]
    ].head()
)

                  name         categories     price  rating  \
0     مطعم وقت الشواية  afghan restaurant  Moderate     NaN   
1         ديوانية عكاظ               café     Cheap     NaN   
2  شاهي جمر راعي الجمس        coffee shop     Cheap     NaN   
3           غدير الشام  afghan restaurant  Moderate     NaN   
4              Dunkin'         donut shop     Cheap     8.9   

   cuisine_similarity  budget_match  rating_match  
0            0.000000             0             0  
1            0.000000             1             0  
2            1.000000             1             0  
3            0.000000             0             0  
4            0.270725             1             1  


In [11]:
#9.Creating user profiles because the dataset has no real user feedback

user_profiles = pd.DataFrame({
    "cuisine": [
        "coffee shop",
        "burger joint",
        "pizza place",
        "indian restaurant",
        "dessert shop",
        "fast food restaurant",
        "bakery",
        "shawarma place",
        "café",
        "middle eastern restaurant"
    ],

    "budget": [
        "Cheap",
        "Moderate",
        "Cheap",
        "Moderate",
        "Cheap",
        "Cheap",
        "Moderate",
        "Cheap",
        "Cheap",
        "Expensive"
    ],

    "minimum_rating": [
        7,
        8,
        7,
        7,
        8,
        6,
        7,
        7,
        8,
        8
    ]
})

user_profiles["user_id"] = range(
    len(user_profiles)
)

print(user_profiles)



                     cuisine     budget  minimum_rating  user_id
0                coffee shop      Cheap               7        0
1               burger joint   Moderate               8        1
2                pizza place      Cheap               7        2
3          indian restaurant   Moderate               7        3
4               dessert shop      Cheap               8        4
5       fast food restaurant      Cheap               6        5
6                     bakery   Moderate               7        6
7             shawarma place      Cheap               7        7
8                       café      Cheap               8        8
9  middle eastern restaurant  Expensive               8        9


In [12]:
#10.Generate Simulated User Interaction Data
# We simulate whether a user likes a restaurant based on
# the match between their preferences and the restaurant.


training_examples = []

for user_id, user in user_profiles.iterrows():

    user_cuisine = tfidf.transform(
        [user["cuisine"]]
    )

    similarities = cosine_similarity(
        user_cuisine,
        tfidf_matrix
    ).flatten()

    sample_indices = np.random.choice(
        len(restaurants),
        size=500,
        replace=False
    )

    for restaurant_index in sample_indices:

        restaurant = restaurants.iloc[
            restaurant_index
        ]

        # Content similarity
        cuisine_similarity = similarities[
            restaurant_index
        ]

        # Budget match
        budget_match = int(
            restaurant["price"] == user["budget"]
        )

        # Rating match
        rating_match = int(
            pd.notna(restaurant["rating"]) and
            restaurant["rating"] >= user["minimum_rating"]
        )

        # Creating a preference score
        preference_score = (
            0.5 * cuisine_similarity +
            0.3 * budget_match +
            0.2 * rating_match
        )

        # Simulate user feedback.

        # WHere higher preference score = higher prob. user likes restaurant

        like_probability = (
            0.85 * preference_score + 0.05
        )

        like_probability = np.clip(
            like_probability,
            0.05,
            0.95
        )

        liked = np.random.binomial(
            1,
            like_probability
        )

        training_examples.append({
            "user_id": user_id,
            "restaurant_index": restaurant_index,
            "cuisine_similarity": cuisine_similarity,
            "budget_match": budget_match,
            "rating_match": rating_match,
            "rating": restaurant["rating"],
            "liked": liked
        })


# Converting interaction examples into a DataFrame
training_data = pd.DataFrame(
    training_examples
)

print("Training data shape:", training_data.shape)

training_data.head()

Training data shape: (5000, 7)


,user_id,restaurant_index,cuisine_similarity,budget_match,rating_match,rating,liked
0,0,9671,0.0,1,0,NaN,0
1,0,13847,0.0,1,0,NaN,0
2,0,19282,0.0,1,0,NaN,0
3,0,11596,0.0,1,0,NaN,0
4,0,5466,0.0,0,0,6.4,0


In [13]:
#11
print("User interaction distribution:")
print(
    training_data["liked"].value_counts()
)

print("\nPercentage of liked interactions:")
print(
    training_data["liked"].mean()
)

User interaction distribution:
liked
0    3748
1    1252
Name: count, dtype: int64

Percentage of liked interactions:
0.2504


In [14]:
#12.Preparing Features and Target

X = training_data[
    [
        "cuisine_similarity",
        "budget_match",
        "rating_match"
    ]
]

# Target:
# 1 = user liked restaurant
# 0 = user did not like restaurant

y = training_data["liked"]

In [15]:
#13.Splitting data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4000
Testing samples: 1000


In [16]:
#14.Train Logistic Regression(baseline model)

logistic_model = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train
)

# Predicting the classes
logistic_predictions = logistic_model.predict(
    X_test
)

# Predicting probabilities
logistic_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

In [17]:
#15.Evaluating Logistic Regression

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_precision = precision_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_predictions,
    zero_division=0
)

logistic_auc = roc_auc_score(
    y_test,
    logistic_probabilities
)

print("Logistic Regression Results")
print("--------------------------------")
print("Accuracy :", logistic_accuracy)
print("Precision:", logistic_precision)
print("Recall   :", logistic_recall)
print("ROC-AUC  :", logistic_auc)

Logistic Regression Results
--------------------------------
Accuracy : 0.669
Precision: 0.4171779141104294
Recall   : 0.816
ROC-AUC  : 0.78104


In [18]:
#16.Training Random Forest(comparison)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

random_forest_model.fit(
    X_train,
    y_train
)

# Predicting the classes
random_forest_predictions = (
    random_forest_model.predict(X_test)
)

# Predicting probabilities
random_forest_probabilities = (
    random_forest_model.predict_proba(X_test)[:, 1]
)

In [19]:
#17.Evaluating Random Forest

random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions
)

random_forest_precision = precision_score(
    y_test,
    random_forest_predictions,
    zero_division=0
)

random_forest_recall = recall_score(
    y_test,
    random_forest_predictions,
    zero_division=0
)

random_forest_auc = roc_auc_score(
    y_test,
    random_forest_probabilities
)

print("Random Forest Results")
print("--------------------------------")
print("Accuracy :", random_forest_accuracy)
print("Precision:", random_forest_precision)
print("Recall   :", random_forest_recall)
print("ROC-AUC  :", random_forest_auc)

Random Forest Results
--------------------------------
Accuracy : 0.656
Precision: 0.4012605042016807
Recall   : 0.764
ROC-AUC  : 0.7373920000000002


In [20]:
#18.Model COmparison

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        logistic_accuracy,
        random_forest_accuracy
    ],

    "Precision": [
        logistic_precision,
        random_forest_precision
    ],

    "Recall": [
        logistic_recall,
        random_forest_recall
    ],

    "ROC-AUC": [
        logistic_auc,
        random_forest_auc
    ]
})

print(model_comparison)

                 Model  Accuracy  Precision  Recall   ROC-AUC
0  Logistic Regression     0.669   0.417178   0.816  0.781040
1        Random Forest     0.656   0.401261   0.764  0.737392


In [21]:
#19.Selecting the best model
# ROC-AUC is used as the main comparison metric.

if random_forest_auc > logistic_auc:

    best_model = random_forest_model
    best_model_name = "Random Forest"

else:

    best_model = logistic_model
    best_model_name = "Logistic Regression"


print("Selected model:", best_model_name)



Selected model: Logistic Regression


In [22]:
#20

user_cuisine = tfidf.transform(
    [user_preferences["cuisine"]]
)

restaurants["cuisine_similarity"] = (
    cosine_similarity(
        user_cuisine,
        tfidf_matrix
    ).flatten()
)

restaurants["budget_match"] = (
    restaurants["price"] ==
    user_preferences["budget"]
).astype(int)

restaurants["rating_match"] = (
    restaurants["rating"].notna() &
    (
        restaurants["rating"] >=
        user_preferences["minimum_rating"]
    )
).astype(int)


# Select the same features used during training

recommendation_features = restaurants[
    [
        "cuisine_similarity",
        "budget_match",
        "rating_match"
    ]
]


# Predict probability that the user will like
# each restaurant.

restaurants["like_probability"] = (
    best_model.predict_proba(
        recommendation_features
    )[:, 1]
)

In [23]:
#21.Filtering restaurants based on user preferences

filtered_recommendations = restaurants[
    (restaurants["price"] ==
     user_preferences["budget"]) &

    (
        restaurants["rating"].notna()
    ) &

    (
        restaurants["rating"] >=
        user_preferences["minimum_rating"]
    )
].copy()

In [24]:
#22. Ranking the restaurants

recommendations = (
    filtered_recommendations
    .sort_values(
        by="like_probability",
        ascending=False
    )
)

In [25]:
#23.Top 10 Recommendations

top_recommendations = recommendations[
    [
        "name",
        "categories",
        "price",
        "rating",
        "cuisine_similarity",
        "like_probability"
    ]
].head(10)

print("Top 10 Recommended Restaurants:")
print(top_recommendations)

Top 10 Recommended Restaurants:
                                 name   categories  price  rating  \
13                          Starbucks  coffee shop  Cheap     8.1   
19213     Wayne's Coffee (واينز كوفي)  coffee shop  Cheap     8.7   
38                          Starbucks  coffee shop  Cheap     8.0   
19206           4Twins Coffee & Sweet  coffee shop  Cheap     8.8   
19204             Starbucks (ستاربكس)  coffee shop  Cheap     9.2   
19173                       Starbucks  coffee shop  Cheap     8.2   
205                          ستار بكس  coffee shop  Cheap     7.4   
62     SADIG Coffee Shop (صادق كافيه)  coffee shop  Cheap     7.5   
123         Coffee Gram (جرام القهوة)  coffee shop  Cheap     8.9   
91                Starbucks (ستاربكس)  coffee shop  Cheap     8.9   

       cuisine_similarity  like_probability  
13                    1.0          0.964134  
19213                 1.0          0.964134  
38                    1.0          0.964134  
19206                 1.

In [26]:
#24.Creating a recommendation function

def recommend_restaurants(
    cuisine,
    budget=None,
    minimum_rating=None,
    n=10
):

    # Convert user's cuisine preference to TF-IDF
    user_cuisine = tfidf.transform(
        [cuisine]
    )

    # Calculate cuisine similarity
    cuisine_similarity = cosine_similarity(
        user_cuisine,
        tfidf_matrix
    ).flatten()

    # Create a copy of the restaurant dataset
    recommendations = restaurants.copy()

    # Store content-based similarity
    recommendations["cuisine_similarity"] = (
        cuisine_similarity
    )

    # Budget match
    if budget is not None:

        recommendations["budget_match"] = (
            recommendations["price"] == budget
        ).astype(int)

    else:

        recommendations["budget_match"] = 0


    # Rating match
    if minimum_rating is not None:

        recommendations["rating_match"] = (
            recommendations["rating"].notna() &
            (
                recommendations["rating"] >=
                minimum_rating
            )
        ).astype(int)

    else:

        recommendations["rating_match"] = 0


    # Prepare features
    features = recommendations[
        [
            "cuisine_similarity",
            "budget_match",
            "rating_match"
        ]
    ]


    # Predict probability of user liking
    recommendations["like_probability"] = (
        best_model.predict_proba(
            features
        )[:, 1]
    )


    # Applying budget filter
    if budget is not None:

        recommendations = recommendations[
            recommendations["price"] == budget
        ]


    # Applying rating filter
    if minimum_rating is not None:

        recommendations = recommendations[
            recommendations["rating"].notna()
        ]

        recommendations = recommendations[
            recommendations["rating"] >= minimum_rating
        ]


    # Ranking by predicted like probability
    recommendations = recommendations.sort_values(
        by="like_probability",
        ascending=False
    )


    # Return top N restaurants
    return recommendations[
        [
            "name",
            "categories",
            "price",
            "rating",
            "cuisine_similarity",
            "like_probability"
        ]
    ].head(n)

In [27]:
#25.Testing the Recommendation System

pizza_recommendations = recommend_restaurants(
    cuisine="pizza place",
    budget="Cheap",
    minimum_rating=7,
    n=10
)

print("Pizza Recommendations:")
print(pizza_recommendations)

Pizza Recommendations:
                                   name   categories  price  rating  \
5281   Oregano Pizzeria Rawabi, Exit 15  pizza place  Cheap     9.4   
10336                    Domino's Pizza  pizza place  Cheap     7.3   
8533       dominos pizza (دومينز بيتزا)  pizza place  Cheap     7.4   
11831    Domino's Pizza (دومينوز بيتزا)  pizza place  Cheap     8.0   
4025                     Domino's Pizza  pizza place  Cheap     7.2   
11824     Maestro Pizza (مايسترو بيتزا)  pizza place  Cheap     7.0   
5864               Pizza Hut (بيتزا هت)  pizza place  Cheap     7.0   
8959      Maestro Pizza (مايسترو بيتزا)  pizza place  Cheap     7.3   
6721              Pizza Era (بيتزا إرا)  pizza place  Cheap     8.5   
4023                     Domino's Pizza  pizza place  Cheap     7.4   

       cuisine_similarity  like_probability  
5281                  1.0          0.964134  
10336                 1.0          0.964134  
8533                  1.0          0.964134  
11831      

In [28]:
#26.Testing another user
burger_recommendations = recommend_restaurants(
    cuisine="burger joint",
    budget="Moderate",
    minimum_rating=8,
    n=10
)

print("Burger Recommendations:")
print(burger_recommendations)

Burger Recommendations:
                                                    name    categories  \
18768                            Burgerizzer (برغرايززر)  burger joint   
17573  Burger & Burger (Burger & Burger | برجر آند برجر)  burger joint   
3233                                           Highway 5  burger joint   
7412          TSC The Sandwich Co. (ذا ساندويتش كومباني)  burger joint   
13880                      Johnny Rocket's (جوني روكيتس)  burger joint   
16072                                12 Burger (١٢ برجر)  burger joint   
7937                           Burger Castle (برجر كاسل)  burger joint   
14805                             Burgerizzr (برغرايززر)  burger joint   
8681                              Hamburgini (هامبرغيني)  burger joint   
6824                              Burgerizzr (برغرايززر)  burger joint   

          price  rating  cuisine_similarity  like_probability  
18768  Moderate     8.2                 1.0          0.964134  
17573  Moderate     8.2          

In [29]:
#27.Testing another user

coffee_recommendations = recommend_restaurants(
    cuisine="coffee shop",
    budget="Cheap",
    minimum_rating=7,
    n=10
)

print("Coffee Shop Recommendations:")
print(coffee_recommendations)

Coffee Shop Recommendations:
                                 name   categories  price  rating  \
13                          Starbucks  coffee shop  Cheap     8.1   
19213     Wayne's Coffee (واينز كوفي)  coffee shop  Cheap     8.7   
38                          Starbucks  coffee shop  Cheap     8.0   
19206           4Twins Coffee & Sweet  coffee shop  Cheap     8.8   
19204             Starbucks (ستاربكس)  coffee shop  Cheap     9.2   
19173                       Starbucks  coffee shop  Cheap     8.2   
205                          ستار بكس  coffee shop  Cheap     7.4   
62     SADIG Coffee Shop (صادق كافيه)  coffee shop  Cheap     7.5   
123         Coffee Gram (جرام القهوة)  coffee shop  Cheap     8.9   
91                Starbucks (ستاربكس)  coffee shop  Cheap     8.9   

       cuisine_similarity  like_probability  
13                    1.0          0.964134  
19213                 1.0          0.964134  
38                    1.0          0.964134  
19206                 1.0  

In [30]:
#28.Saving the Trained Recommendation Pipeline

pipeline_artifacts = {

    "model": best_model,

    "model_name": best_model_name,

    "vectorizer": tfidf,

    "restaurants_df": restaurants,

    "features": [
        "cuisine_similarity",
        "budget_match",
        "rating_match"
    ]
}

joblib.dump(
    pipeline_artifacts,
    "riyadh_recommender_pipeline.pkl"
)

print(
    "Recommendation pipeline is saved"
)

Recommendation pipeline is saved


In [31]:
from google.colab import files
files.download("riyadh_recommender_pipeline.pkl")


from google.colab import files
restaurants.to_csv("riyadh_restaurants_clean.csv", index=False)
files.download("riyadh_restaurants_clean.csv")



from google.colab import files
model_comparison.to_csv("model_performance.csv", index=False)
files.download("model_performance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>